# 01 Momentum Research — Macro Metals System

> **Strategy:** Canonical Time-Series Momentum (TSMOM)
> **Reference:** Moskowitz, Ooi & Pedersen (2012); Quantpedia; JPM/CME report
> **Scope:** In-sample development (2015–2022), monthly rebalance
> **Universe:** Multi-asset futures — commodities (70% risk budget), rates, equities, FX (30%)
> **Signal:** sign(12-month return), monthly rebalance, inverse-vol position sizing
> **Overlay:** Sector risk budgets + portfolio-level vol targeting (10% annual)
>
> Self-contained BQuant notebook — executable top-to-bottom.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import yaml
from pathlib import Path
from datetime import datetime

# Bloomberg BQL
import bql
bq = bql.Service()

print(f"Session started : {datetime.now():%Y-%m-%d %H:%M}")
print(f"BQL service     : {type(bq).__name__}")
print(f"NumPy {np.__version__}  |  pandas {pd.__version__}")

## Config & Parameters

Load `parameters.yaml` and `tickers.yaml`.

| Parameter | Value | Description |
|-----------|-------|-------------|
| `lookback_days` | 252 | 12-month return lookback |
| `rebalance_freq` | monthly | Signal + vol + weights frozen monthly |
| `target_vol_annual` | 0.10 | Per-instrument vol target |
| `vol_decay_lambda` | 0.94 | EWMA decay (RiskMetrics) |
| `leverage_cap_multiplier` | 2.0 | Max weight scaling |
| `tc_bp_per_side` | 2.0 | Transaction cost per side |
| `commodities_risk_budget` | 0.70 | Commodity sector risk share |
| `diversifiers_risk_budget` | 0.30 | Rates + equities + FX risk share |
| `portfolio_vol_target` | 0.10 | Portfolio-level vol overlay |

In [ ]:
CONFIG_DIR = Path("config")

with open(CONFIG_DIR / "parameters.yaml") as f:
    params = yaml.safe_load(f)
with open(CONFIG_DIR / "tickers.yaml") as f:
    tickers = yaml.safe_load(f)

gcfg    = params["global"]
targets = params["performance_targets"]

# ── Canonical TSMOM parameters ───────────────────────────────────
def _get(cfg, key, default, label=""):
    if key in cfg:
        return cfg[key]
    print(f"  Warning: {label or key} missing from config, using default {default}")
    return default

LOOKBACK_DAYS = 252
TARGET_VOL    = _get(gcfg, "target_portfolio_vol_annual", 0.10, "target_vol_annual")
VOL_LAMBDA    = _get(gcfg, "vol_decay_lambda", 0.94)
LEV_CAP       = _get(gcfg, "vol_cap_multiplier", 2.0, "leverage_cap_multiplier")
TC_BP         = 2.0

# Sector risk budgets (commodity-focused)
COMMODITY_RISK_BUDGET    = 0.70
DIVERSIFIER_RISK_BUDGET  = 0.30  # rates + equities + fx
PORTFOLIO_VOL_TARGET     = 0.10
PORTFOLIO_VOL_LOOKBACK   = 60    # days for rolling portfolio vol
PORTFOLIO_SCALE_BOUNDS   = (0.5, 2.0)

IS_START = gcfg["in_sample_start"]
IS_END   = gcfg["in_sample_end"]

print("Canonical TSMOM parameters:")
print(f"  Lookback           : {LOOKBACK_DAYS}d (12 months)")
print(f"  Rebalance          : monthly")
print(f"  Per-inst vol target: {TARGET_VOL:.0%}")
print(f"  EWMA lambda        : {VOL_LAMBDA}")
print(f"  Leverage cap       : {LEV_CAP:.1f}x")
print(f"  TC per side        : {TC_BP:.1f} bp")
print(f"  Risk budget        : {COMMODITY_RISK_BUDGET:.0%} commodities / {DIVERSIFIER_RISK_BUDGET:.0%} diversifiers")
print(f"  Portfolio vol tgt  : {PORTFOLIO_VOL_TARGET:.0%}")
print(f"  IS period          : {IS_START} to {IS_END}")

## Universe Expansion

Multi-asset futures universe in 4 buckets, matching the canonical TSMOM literature
(Moskowitz et al. documented 58 instruments across equity index, currency, commodity,
and bond futures).

| Bucket | Role | Risk Budget |
|--------|------|-------------|
| **Commodities** | Precious metals, energy, agriculture, base metals | 70% |
| **Rates** | SOFR strip + UST futures | 30% combined |
| **Equities** | Major equity index futures | |
| **FX** | G10 liquid pairs | |

Tickers resolved from `tickers.yaml`; missing instruments logged and skipped.

In [ ]:
# ── Multi-asset TSMOM universe ────────────────────────────────────
universe = {
    "commodities": [
        # Precious metals (core)
        "gc_fut_front", "si_fut_front", "pl_fut_front",
        # Base metals
        "hg_fut_front",
        # Energy
        "cl_fut_front", "ng_fut_front",
        # Agriculture
        "w_fut_front", "c_fut_front", "s_fut_front",
    ],
    "rates": [
        # SOFR futures strip
        "sofr_fut_front", "sofr_fut_second", "sofr_fut_third", "sofr_fut_fourth",
        # UST futures
        "ust_2y_fut", "ust_5y_fut", "ust_10y_fut", "ust_30y_fut",
    ],
    "equities": [
        "es_fut_front", "nq_fut_front", "stoxx50_fut_front", "nikkei_fut_front",
    ],
    "fx": [
        "eurusd_spot", "usdjpy_spot", "gbpusd_spot", "audusd_spot",
        "nzdusd_spot", "usdcad_spot",
    ],
}

# Human-readable labels
LABELS = {
    "gc_fut_front": "Gold (GC)", "si_fut_front": "Silver (SI)",
    "pl_fut_front": "Platinum (PL)", "hg_fut_front": "Copper (HG)",
    "cl_fut_front": "Crude Oil (CL)", "ng_fut_front": "Nat Gas (NG)",
    "w_fut_front": "Wheat (W)", "c_fut_front": "Corn (C)", "s_fut_front": "Soybeans (S)",
    "sofr_fut_front": "SOFR 1st", "sofr_fut_second": "SOFR 2nd",
    "sofr_fut_third": "SOFR 3rd", "sofr_fut_fourth": "SOFR 4th",
    "ust_2y_fut": "UST 2Y (TU)", "ust_5y_fut": "UST 5Y (FV)",
    "ust_10y_fut": "UST 10Y (TY)", "ust_30y_fut": "UST 30Y (US)",
    "es_fut_front": "S&P 500 (ES)", "nq_fut_front": "Nasdaq (NQ)",
    "stoxx50_fut_front": "EuroStoxx 50", "nikkei_fut_front": "Nikkei 225",
    "eurusd_spot": "EURUSD", "usdjpy_spot": "USDJPY", "gbpusd_spot": "GBPUSD",
    "audusd_spot": "AUDUSD", "nzdusd_spot": "NZDUSD", "usdcad_spot": "USDCAD",
}

# Map instrument -> bucket for risk budgeting
INST_BUCKET = {}
for bucket, instruments in universe.items():
    for inst in instruments:
        INST_BUCKET[inst] = bucket

# ── Resolve tickers from YAML ────────────────────────────────────
def resolve_ticker(logical_name: str, ticker_map: dict) -> str:
    """Resolve logical name to Bloomberg ticker; return None if missing."""
    for group in ticker_map.values():
        if isinstance(group, dict) and logical_name in group:
            return group[logical_name]
    return None

resolved = {}
missing = []
for bucket, instruments in universe.items():
    for inst in instruments:
        bbg = resolve_ticker(inst, tickers)
        if bbg:
            resolved[inst] = bbg
        else:
            missing.append(inst)

print(f"Universe: {sum(len(v) for v in universe.values())} instruments across {len(universe)} buckets")
print(f"Resolved: {len(resolved)} | Missing: {len(missing)}")
if missing:
    print(f"\n  Missing tickers (will be skipped): {missing}")

print("\nBucket breakdown:")
for bucket, instruments in universe.items():
    n_resolved = sum(1 for i in instruments if i in resolved)
    print(f"  {bucket:15s}: {n_resolved}/{len(instruments)} resolved")

## Data Pipeline (BQL)

Fetch daily `PX_LAST` for the full multi-asset universe.
Uses corrected BQL syntax — `df.set_index('DATE')`.
Instruments with < 252+30 days of history are dropped.

In [ ]:
class BQuantDataLoader:
    """Fetch historical prices via Bloomberg BQL."""

    def __init__(self, ticker_map: dict) -> None:
        self._tickers = ticker_map
        self._bq = bql.Service()

    def resolve(self, logical_name: str) -> str:
        for group in self._tickers.values():
            if isinstance(group, dict) and logical_name in group:
                return group[logical_name]
        raise KeyError(f"{logical_name} not in tickers.yaml")

    def get_history(self, logical_name: str, start: str, end: str,
                    field: str = "PX_LAST") -> pd.Series:
        bbg = self.resolve(logical_name)
        request = bql.Request(
            bbg,
            {field: self._bq.data.px_last(
                dates=self._bq.func.range(start, end)
            )},
        )
        try:
            response = self._bq.execute(request)
            df = response[0].df()
            if df.empty:
                return pd.Series(dtype=float, name=logical_name)
            df_fixed = df.set_index('DATE')
            series = df_fixed[field]
            series.index = pd.to_datetime(series.index, errors='coerce')
            series = series.dropna()
            series = series[~series.index.duplicated(keep='last')]
            series = series.sort_index().astype(float)
            series.name = logical_name
            series.index.name = "date"
            if not isinstance(series.index, pd.DatetimeIndex):
                series.index = pd.to_datetime(series.index)
            return series
        except Exception as exc:
            print(f"  ! {logical_name} ({bbg}): {exc}")
            return pd.Series(dtype=float, name=logical_name)


# ── Fetch all resolved instruments ────────────────────────────────
loader = BQuantDataLoader(tickers)
prices_raw = {}
fetch_status = []

for inst in resolved:
    label = LABELS.get(inst, inst)
    bucket = INST_BUCKET[inst]
    print(f"  {label:22s}", end=" ")
    s = loader.get_history(inst, IS_START, IS_END)
    n_obs = len(s)
    if n_obs > 0:
        prices_raw[inst] = s
        print(f"OK  {n_obs:>5d} obs  [{s.index[0]:%Y-%m-%d} -> {s.index[-1]:%Y-%m-%d}]")
    else:
        print("MISSING")
    fetch_status.append({"instrument": inst, "label": label, "bucket": bucket,
                         "obs": n_obs, "status": "OK" if n_obs > 0 else "MISSING"})

# Build aligned panel
prices_df = pd.DataFrame(prices_raw).sort_index()
prices_df = prices_df[prices_df.index.notna()].ffill()

# Drop instruments with insufficient history
MIN_OBS = LOOKBACK_DAYS + 30
sufficient = prices_df.count() >= MIN_OBS
dropped_insts = prices_df.columns[~sufficient].tolist()
if dropped_insts:
    print(f"\nDropped (< {MIN_OBS} obs): {[LABELS.get(i,i) for i in dropped_insts]}")
prices_df = prices_df[prices_df.columns[sufficient]]

# Update INST_BUCKET to only include surviving instruments
active_instruments = list(prices_df.columns)
active_buckets = {}
for inst in active_instruments:
    b = INST_BUCKET.get(inst)
    if b:
        active_buckets.setdefault(b, []).append(inst)

print(f"\nFinal panel: {prices_df.shape[0]} days x {prices_df.shape[1]} instruments")

# ── Breadth report ────────────────────────────────────────────────
print("\n" + "=" * 65)
print("  BREADTH REPORT")
print("=" * 65)
status_df = pd.DataFrame(fetch_status)
for bucket in universe:
    bdf = status_df[status_df["bucket"] == bucket]
    n_total = len(bdf)
    n_ok = (bdf["status"] == "OK").sum()
    n_sufficient = sum(1 for i in universe[bucket] if i in prices_df.columns)
    print(f"  {bucket:15s}: {n_total} defined | {n_ok} fetched | {n_sufficient} with >= {MIN_OBS}d history")

# Missing data counts per instrument
miss_counts = prices_df.isna().sum()
if miss_counts.sum() > 0:
    print("\nMissing data counts (after ffill):")
    for inst in prices_df.columns:
        m = miss_counts[inst]
        if m > 0:
            print(f"  {LABELS.get(inst, inst):22s}: {m}")
else:
    print("\nNo missing data after forward-fill.")

prices_df.tail(3)

## Canonical TSMOM Signal

**Moskowitz, Ooi & Pedersen (2012):**
- Compute 12-month (252 trading day) return for each instrument
- Signal = `sign(return)`: +1 long, -1 short
- Resample to month-end, shift by 1 month (no lookahead)
- Forward-fill within each month → constant intra-month signal

No extra filters, no dead zone, no EMA, no Donchian, no stops.

In [ ]:
class CanonicalTSMOMStrategy:
    """Canonical TSMOM: sign(12M return), monthly rebalance."""

    def __init__(self, lookback_days: int = 252) -> None:
        self.lookback_days = lookback_days

    def compute_monthly_signal(self, prices_df: pd.DataFrame) -> pd.DataFrame:
        """Monthly-rebalanced sign(12M return) signal.

        Returns DataFrame of daily signals in {-1, 0, +1}.
        """
        # 12-month return
        r12 = prices_df.pct_change(self.lookback_days)
        sig_daily = np.sign(r12)

        # Avoid lookahead: shift by 1 day
        sig_daily = sig_daily.shift(1)

        # Month-end snapshot, then shift by 1 month
        signal_monthly = sig_daily.resample("M").last()
        signal_monthly = signal_monthly.shift(1)

        # Forward-fill to daily
        signal_daily = signal_monthly.reindex(prices_df.index).ffill().fillna(0.0)

        return signal_daily


tsmom = CanonicalTSMOMStrategy(lookback_days=LOOKBACK_DAYS)
signals = tsmom.compute_monthly_signal(prices_df)

print(f"Signal matrix: {signals.shape[0]} days x {signals.shape[1]} instruments")

# ── Sanity: signal distribution ──────────────────────────────────
print("\n" + "=" * 65)
print("  SIGNAL DISTRIBUTION (% of days)")
print("=" * 65)
for col in signals.columns:
    s = signals[col]
    n = len(s)
    pct_long  = (s ==  1).sum() / n * 100
    pct_short = (s == -1).sum() / n * 100
    pct_flat  = (s ==  0).sum() / n * 100
    label = LABELS.get(col, col)
    print(f"  {label:22s}  Long={pct_long:5.1f}%  Short={pct_short:5.1f}%  Flat={pct_flat:5.1f}%")

# ── Sanity: average holding period ───────────────────────────────
print("\n" + "=" * 65)
print("  AVERAGE HOLDING PERIOD")
print("=" * 65)
for col in signals.columns:
    changes = (signals[col].diff().abs() > 0).sum()
    avg_hold = len(signals[col]) / max(changes, 1)
    label = LABELS.get(col, col)
    print(f"  {label:22s}  {avg_hold:.0f} days  ({avg_hold/21:.1f} months)")

# Strict check
unique_vals = set()
for col in signals.columns:
    unique_vals.update(signals[col].dropna().unique())
print(f"\nUnique signal values: {sorted(unique_vals)}")
assert unique_vals <= {-1.0, 0.0, 1.0}, "Signal values outside {-1, 0, +1}!"
print("Signal check passed: strictly in {-1, 0, +1}")

## Monthly Frozen Vol Scaling

**EWMA volatility** (λ = 0.94, RiskMetrics standard):
- `alpha = 1 - lambda` on squared returns → sqrt → annualise
- Sample at month-end, shift by 1 month, forward-fill intra-month

**Weights:**
- `raw_w = signal × (target_vol / vol_frozen)`
- Per-instrument cap: `±2.0 × (target_vol / vol_frozen)`

All weights frozen within each month — turnover only at month boundaries.

In [ ]:
def ex_ante_vol_ewma(returns: pd.DataFrame, lam: float = 0.94) -> pd.DataFrame:
    """EWMA annualised volatility (RiskMetrics style).

    Uses alpha = 1 - lambda on squared returns.
    """
    alpha = 1 - lam
    ewma_var = returns.pow(2).ewm(alpha=alpha, adjust=False).mean()
    vol_daily = np.sqrt(ewma_var)
    vol_ann = vol_daily * np.sqrt(252)
    return vol_ann


# Daily returns and vol
ret = prices_df.pct_change().fillna(0.0)
vol_ann = ex_ante_vol_ewma(ret, lam=VOL_LAMBDA)

# ── Month-end vol snapshot, shifted by 1 month ───────────────────
vol_monthly = vol_ann.resample("M").last().shift(1)
vol_frozen_daily = vol_monthly.reindex(prices_df.index).ffill()

# Replace zero/NaN vol
vol_frozen_daily = vol_frozen_daily.replace(0, np.nan)

# ── Raw weights ──────────────────────────────────────────────────
raw_w = signals * (TARGET_VOL / vol_frozen_daily)

# ── Per-instrument leverage cap ──────────────────────────────────
# Cap at ±LEV_CAP × (target / vol), same as ±LEV_CAP in weight space
w_cap = LEV_CAP * (TARGET_VOL / vol_frozen_daily)
weights = raw_w.clip(-w_cap, w_cap).fillna(0.0)

# ── Assertion: weights only change monthly ───────────────────────
print("=" * 65)
print("  WEIGHT CHANGE FREQUENCY (should be ~12/year = ~96 total)")
print("=" * 65)
for col in weights.columns:
    changes = (weights[col].diff().abs() > 1e-10).sum()
    pct = changes / len(weights) * 100
    label = LABELS.get(col, col)
    print(f"  {label:22s}  {changes:4d} changes  ({pct:.1f}% of days)")

# Average vol by instrument
print("\n" + "=" * 65)
print("  AVERAGE ANNUALISED VOL (EWMA, frozen)")
print("=" * 65)
for col in vol_frozen_daily.columns:
    avg = vol_frozen_daily[col].dropna().mean()
    label = LABELS.get(col, col)
    print(f"  {label:22s}  {avg:.1%}")

## Sector Risk Budgets (Commodity-Focused)

Scale weights so the momentum sleeve stays commodity-focused:
- **Commodities:** 70% of portfolio risk
- **Diversifiers** (rates + equities + FX): 30% of portfolio risk

Risk per bucket ≈ `Σ |w_i| × vol_i` within bucket.
Bucket scaling computed monthly (month-end) and held constant intra-month.

In [ ]:
def compute_bucket_scaling(
    weights: pd.DataFrame,
    vol_frozen: pd.DataFrame,
    inst_bucket: dict,
    commodity_budget: float = 0.70,
    diversifier_budget: float = 0.30,
) -> pd.DataFrame:
    """Compute monthly bucket-level scaling factors.

    Returns DataFrame of per-instrument scaling factors (daily, frozen monthly).
    """
    # Bucket risk: sum of |w_i| * vol_i within bucket
    commodity_insts = [i for i in weights.columns if inst_bucket.get(i) == "commodities"]
    diversifier_insts = [i for i in weights.columns if inst_bucket.get(i) != "commodities"]

    # Daily bucket risk (before scaling)
    risk_comm = (weights[commodity_insts].abs() * vol_frozen[commodity_insts]).sum(axis=1) if commodity_insts else pd.Series(0.0, index=weights.index)
    risk_div  = (weights[diversifier_insts].abs() * vol_frozen[diversifier_insts]).sum(axis=1) if diversifier_insts else pd.Series(0.0, index=weights.index)
    total_risk = risk_comm + risk_div

    # Target risk per bucket
    target_comm = total_risk * commodity_budget
    target_div  = total_risk * diversifier_budget

    # Scaling factors
    scale_comm = (target_comm / risk_comm.replace(0, np.nan)).fillna(1.0)
    scale_div  = (target_div / risk_div.replace(0, np.nan)).fillna(1.0)

    # Freeze monthly: sample at month-end, shift, ffill
    scale_comm_m = scale_comm.resample("M").last().shift(1)
    scale_div_m  = scale_div.resample("M").last().shift(1)
    scale_comm_d = scale_comm_m.reindex(weights.index).ffill().fillna(1.0)
    scale_div_d  = scale_div_m.reindex(weights.index).ffill().fillna(1.0)

    # Build per-instrument scaling
    scaling = pd.DataFrame(1.0, index=weights.index, columns=weights.columns)
    for inst in commodity_insts:
        scaling[inst] = scale_comm_d
    for inst in diversifier_insts:
        scaling[inst] = scale_div_d

    return scaling


bucket_scaling = compute_bucket_scaling(
    weights, vol_frozen_daily, INST_BUCKET,
    commodity_budget=COMMODITY_RISK_BUDGET,
    diversifier_budget=DIVERSIFIER_RISK_BUDGET,
)

weights_scaled = weights * bucket_scaling

# ── Diagnostics: realised risk share by bucket ───────────────────
commodity_insts = [i for i in weights_scaled.columns if INST_BUCKET.get(i) == "commodities"]
diversifier_insts = [i for i in weights_scaled.columns if INST_BUCKET.get(i) != "commodities"]

risk_comm_post = (weights_scaled[commodity_insts].abs() * vol_frozen_daily[commodity_insts]).sum(axis=1)
risk_div_post  = (weights_scaled[diversifier_insts].abs() * vol_frozen_daily[diversifier_insts]).sum(axis=1)
total_risk_post = risk_comm_post + risk_div_post

# Store for plotting later
bucket_risk_shares = pd.DataFrame({
    "commodities_pct": (risk_comm_post / total_risk_post.replace(0, np.nan) * 100).fillna(0),
    "diversifiers_pct": (risk_div_post / total_risk_post.replace(0, np.nan) * 100).fillna(0),
})

avg_comm_share = bucket_risk_shares["commodities_pct"].mean()
avg_div_share  = bucket_risk_shares["diversifiers_pct"].mean()

print("=" * 65)
print("  REALISED RISK SHARE BY BUCKET (after scaling)")
print("=" * 65)
print(f"  Commodities  : {avg_comm_share:.1f}%  (target: {COMMODITY_RISK_BUDGET*100:.0f}%)")
print(f"  Diversifiers : {avg_div_share:.1f}%  (target: {DIVERSIFIER_RISK_BUDGET*100:.0f}%)")

# Per-bucket detail
for bucket in universe:
    insts = [i for i in weights_scaled.columns if INST_BUCKET.get(i) == bucket]
    if insts:
        risk_b = (weights_scaled[insts].abs() * vol_frozen_daily[insts]).sum(axis=1)
        share = (risk_b / total_risk_post.replace(0, np.nan) * 100).mean()
        print(f"    {bucket:15s}: {share:.1f}%  ({len(insts)} instruments)")

## Portfolio Vol Targeting Overlay

After bucket scaling, apply a portfolio-level vol overlay to target 10% annual:
- Compute rolling 60-day realised portfolio vol
- Scale = target / realised, capped to [0.5, 2.0]
- Compute monthly (month-end), shift by 1, forward-fill daily
- Multiply all weights by this single scalar

In [ ]:
# ── Pre-overlay portfolio returns (for vol estimation) ────────────
port_ret_pre = (weights_scaled.shift(1) * ret).sum(axis=1)

# Rolling portfolio vol (annualised)
port_vol_rolling = port_ret_pre.rolling(PORTFOLIO_VOL_LOOKBACK, min_periods=20).std() * np.sqrt(252)

# Scale factor
port_scale = PORTFOLIO_VOL_TARGET / port_vol_rolling.replace(0, np.nan)
port_scale = port_scale.clip(PORTFOLIO_SCALE_BOUNDS[0], PORTFOLIO_SCALE_BOUNDS[1]).fillna(1.0)

# Freeze monthly
port_scale_monthly = port_scale.resample("M").last().shift(1)
port_scale_daily = port_scale_monthly.reindex(prices_df.index).ffill().fillna(1.0)

# Final weights
final_weights = weights_scaled.multiply(port_scale_daily, axis=0)

print("Portfolio vol overlay:")
print(f"  Pre-overlay avg vol : {port_vol_rolling.dropna().mean():.1%}")
print(f"  Avg scale factor    : {port_scale_daily.dropna().mean():.2f}")
print(f"  Scale range         : [{port_scale_daily.min():.2f}, {port_scale_daily.max():.2f}]")

# ── Confirm weights still only change monthly ────────────────────
total_changes = 0
for col in final_weights.columns:
    changes = (final_weights[col].diff().abs() > 1e-10).sum()
    total_changes += changes
avg_changes = total_changes / len(final_weights.columns)
print(f"\n  Avg weight changes per instrument: {avg_changes:.0f}  (expect ~{len(pd.date_range(IS_START, IS_END, freq='M'))} months)")

## Backtest

- `port_ret_gross = Σ (final_weight(t-1) × return(t))`
- Turnover from final weight changes; TC = turnover × bp/10000
- **No division by n_active** — weights already vol-scaled and budget-constrained

In [ ]:
# ── Portfolio returns ──────────────────────────────────────────────
port_ret_gross = (final_weights.shift(1) * ret).sum(axis=1)

# ── Turnover + transaction costs ─────────────────────────────────
turnover_daily = (final_weights - final_weights.shift(1)).abs().sum(axis=1).fillna(0.0)
tc_daily = turnover_daily * (TC_BP / 10_000)
port_ret_net = port_ret_gross - tc_daily
port_equity = 1_000_000 * (1 + port_ret_net).cumprod()

# ── Per-instrument equity curves ─────────────────────────────────
inst_equity = {}
inst_metrics = []
for col in final_weights.columns:
    r_inst = (final_weights[col].shift(1) * ret[col])
    to_inst = final_weights[col].diff().abs().fillna(0.0)
    tc_inst = to_inst * (TC_BP / 10_000)
    r_net = r_inst - tc_inst
    eq = 1_000_000 * (1 + r_net).cumprod()
    inst_equity[col] = eq

    # Metrics
    n_years = len(r_net) / 252
    total = (1 + r_net).prod()
    ann_ret = total ** (1 / max(n_years, 0.01)) - 1
    ann_vol = r_net.std() * np.sqrt(252)
    sharpe = ann_ret / ann_vol if ann_vol > 0 else 0.0
    rm = eq.cummax()
    dd = (eq - rm) / rm
    max_dd = float(-dd.min()) if len(dd) > 0 else 0.0
    calmar = ann_ret / max_dd if max_dd > 0 else 0.0
    hit = float((r_net > 0).sum() / len(r_net)) if len(r_net) > 0 else 0.0
    ann_to = to_inst.sum() / max(n_years, 0.01)

    inst_metrics.append({
        "Instrument": LABELS.get(col, col),
        "Bucket": INST_BUCKET.get(col, "?"),
        "Ann. Return": ann_ret, "Ann. Vol": ann_vol,
        "Sharpe": sharpe, "Max DD": max_dd,
        "Calmar": calmar, "Hit Rate": hit,
        "Ann. Turnover": ann_to,
    })

# ── Portfolio-level metrics ──────────────────────────────────────
r = port_ret_net
eq = port_equity
n_years = len(r) / 252
total = (1 + r).prod()
ann_ret = total ** (1 / max(n_years, 0.01)) - 1
ann_vol = r.std() * np.sqrt(252)
sharpe = ann_ret / ann_vol if ann_vol > 0 else 0.0
rm = eq.cummax()
dd = (eq - rm) / rm
max_dd = float(-dd.min())
calmar = ann_ret / max_dd if max_dd > 0 else 0.0
hit = float((r > 0).sum() / len(r))
ann_to = turnover_daily.sum() / max(n_years, 0.01)

port_metrics = {
    "Instrument": "PORTFOLIO",
    "Bucket": "all",
    "Ann. Return": ann_ret, "Ann. Vol": ann_vol,
    "Sharpe": sharpe, "Max DD": max_dd,
    "Calmar": calmar, "Hit Rate": hit,
    "Ann. Turnover": ann_to,
}

# Build metrics table
metrics_df = pd.DataFrame(inst_metrics + [port_metrics]).set_index("Instrument")

fmt_df = metrics_df.copy()
for c in ["Ann. Return", "Ann. Vol", "Max DD", "Hit Rate"]:
    fmt_df[c] = fmt_df[c].map("{:.1%}".format)
fmt_df["Sharpe"]  = fmt_df["Sharpe"].map("{:.2f}".format)
fmt_df["Calmar"]  = fmt_df["Calmar"].map("{:.2f}".format)
fmt_df["Ann. Turnover"] = fmt_df["Ann. Turnover"].map("{:.1f}x".format)

print("=" * 75)
print("  CANONICAL TSMOM — METRICS (IS 2015-2022)")
print("=" * 75)
fmt_df

## Diagnostics

Turnover sanity, portfolio vol realised, correlation structure.

In [ ]:
# ── Turnover sanity ───────────────────────────────────────────────
print("=" * 65)
print("  TURNOVER DIAGNOSTICS")
print("=" * 65)
print(f"  Total turnover (sum of |dw|)     : {turnover_daily.sum():.2f}")
print(f"  Annualised turnover              : {ann_to:.1f}x")
print(f"  Days with turnover > 0           : {(turnover_daily > 1e-10).sum()} "
      f"of {len(turnover_daily)} ({(turnover_daily > 1e-10).mean():.1%})")
print(f"  Max daily turnover               : {turnover_daily.max():.4f}")

# ── Realised portfolio vol ───────────────────────────────────────
realised_vol = port_ret_net.rolling(60, min_periods=20).std() * np.sqrt(252)
print(f"\n  Realised portfolio vol (60d rolling):")
print(f"    Mean   : {realised_vol.dropna().mean():.1%}")
print(f"    Median : {realised_vol.dropna().median():.1%}")
print(f"    Range  : [{realised_vol.dropna().min():.1%}, {realised_vol.dropna().max():.1%}]")

# ── Cross-instrument return correlations ─────────────────────────
corr = ret[active_instruments].rename(columns=LABELS).corr()
print(f"\n  Average pairwise correlation: {corr.values[np.triu_indices_from(corr.values, k=1)].mean():.3f}")

## Visualisations

1. Portfolio equity (net) + rolling vol
2. Per-instrument equity curves
3. Monthly signal heatmap (+1 / -1)
4. Bucket risk share over time
5. Turnover over time (monthly spikes)

In [ ]:
# --- 1. Portfolio Equity + Rolling Vol ---
fig_port = make_subplots(
    rows=2, cols=1, shared_xaxes=True,
    row_heights=[0.65, 0.35],
    subplot_titles=["Portfolio Equity (net of costs)", "Rolling 60d Vol (annualised)"],
    vertical_spacing=0.08,
)
fig_port.add_trace(go.Scatter(
    x=port_equity.index, y=port_equity,
    name="Portfolio", line=dict(color="#2c3e50", width=2),
    fill="tozeroy", fillcolor="rgba(44, 62, 80, 0.08)",
), row=1, col=1)
fig_port.add_trace(go.Scatter(
    x=realised_vol.index, y=realised_vol,
    name="Realised Vol", line=dict(color="#e67e22", width=1.5),
), row=2, col=1)
fig_port.add_hline(y=PORTFOLIO_VOL_TARGET, line_dash="dash", line_color="red",
                   annotation_text="10% target", row=2, col=1)
fig_port.update_layout(
    title="Canonical TSMOM — Portfolio (IS: 2015-2022, $1M start)",
    template="plotly_white", height=600, hovermode="x unified",
    showlegend=False,
)
fig_port.update_yaxes(title_text="Equity ($)", tickformat="$,.0f", row=1, col=1)
fig_port.update_yaxes(title_text="Vol", tickformat=".0%", row=2, col=1)
fig_port.show()

# --- 2. Per-Instrument Equity Curves ---
eq_df = pd.DataFrame({LABELS.get(k,k): v for k, v in inst_equity.items()})
eq_norm = eq_df / eq_df.iloc[0]

# Color by bucket
bucket_colors = {"commodities": "#f39c12", "rates": "#3498db",
                 "equities": "#2ecc71", "fx": "#9b59b6"}
fig_inst = go.Figure()
for col_orig in final_weights.columns:
    label = LABELS.get(col_orig, col_orig)
    bucket = INST_BUCKET.get(col_orig, "?")
    if label in eq_norm.columns:
        fig_inst.add_trace(go.Scatter(
            x=eq_norm.index, y=eq_norm[label], name=label,
            line=dict(color=bucket_colors.get(bucket, "#95a5a6"), width=1.2),
            legendgroup=bucket, legendgrouptitle_text=bucket,
        ))
fig_inst.update_layout(
    title="Per-Instrument Equity (normalised to $1)",
    template="plotly_white", height=550, hovermode="x unified",
    legend=dict(orientation="h", y=-0.2),
    yaxis_title="Growth of $1", yaxis_tickformat="$.2f",
)
fig_inst.show()

# --- 3. Monthly Signal Heatmap ---
sig_monthly = signals.resample("M").last().rename(columns=LABELS)
fig_heat = px.imshow(
    sig_monthly.T,
    color_continuous_scale=[[0, "#e74c3c"], [0.5, "#ecf0f1"], [1, "#2ecc71"]],
    zmin=-1, zmax=1,
    title="Monthly Signal Regime (+1 Long / -1 Short)",
    labels={"x": "", "y": "Instrument", "color": "Signal"},
    aspect="auto",
)
fig_heat.update_layout(template="plotly_white", height=500,
                       xaxis=dict(dtick="M3", tickformat="%Y-%m"))
fig_heat.show()

# --- 4. Bucket Risk Share Over Time ---
fig_bucket = go.Figure()
fig_bucket.add_trace(go.Scatter(
    x=bucket_risk_shares.index,
    y=bucket_risk_shares["commodities_pct"].rolling(21).mean(),
    name="Commodities", fill="tozeroy",
    line=dict(color="#f39c12"),
))
fig_bucket.add_trace(go.Scatter(
    x=bucket_risk_shares.index,
    y=100,  # not a trace, just for reference
    name="Diversifiers", fill="tonexty",
    line=dict(color="#3498db"),
    # actually plot diversifiers
))
# Redo properly
fig_bucket = go.Figure()
comm_smooth = bucket_risk_shares["commodities_pct"].rolling(21).mean()
fig_bucket.add_trace(go.Scatter(
    x=comm_smooth.index, y=comm_smooth,
    name="Commodities %", line=dict(color="#f39c12", width=2),
))
fig_bucket.add_hline(y=COMMODITY_RISK_BUDGET*100, line_dash="dash",
                     line_color="#f39c12", annotation_text="70% target")
fig_bucket.add_hline(y=DIVERSIFIER_RISK_BUDGET*100, line_dash="dash",
                     line_color="#3498db", annotation_text="30% target")
div_smooth = bucket_risk_shares["diversifiers_pct"].rolling(21).mean()
fig_bucket.add_trace(go.Scatter(
    x=div_smooth.index, y=div_smooth,
    name="Diversifiers %", line=dict(color="#3498db", width=2),
))
fig_bucket.update_layout(
    title="Bucket Risk Share Over Time (21d smoothed)",
    template="plotly_white", height=400, hovermode="x unified",
    yaxis_title="Risk Share (%)", yaxis_range=[0, 100],
)
fig_bucket.show()

# --- 5. Turnover Over Time ---
fig_turn = go.Figure()
fig_turn.add_trace(go.Bar(
    x=turnover_daily.index, y=turnover_daily,
    marker_color="#3498db", opacity=0.7, name="Turnover",
))
fig_turn.update_layout(
    title="Daily Turnover (should spike at month boundaries only)",
    template="plotly_white", height=350,
    yaxis_title="Turnover (sum |Δw|)", hovermode="x unified",
)
fig_turn.show()

## Performance Summary

Compare canonical TSMOM portfolio metrics against memory file targets (§6.3).

In [ ]:
pm = port_metrics

comparison = pd.DataFrame({
    "Metric": [
        "Sharpe Ratio", "Annualised Vol", "Max Drawdown",
        "Calmar Ratio", "Hit Rate", "Ann. Turnover",
    ],
    "Target (section 6.3)": [
        f"> {targets['sharpe_per_strategy']:.1f}",
        f"{targets['vol_range_annual'][0]:.0%} - {targets['vol_range_annual'][1]:.0%}",
        f"< {targets['max_drawdown_pct']:.0f}%",
        f"> {targets['calmar_ratio']:.1f}",
        f"> {targets['hit_rate_daily']:.0%}",
        f"< {targets['max_turnover_annual']:.0f}x",
    ],
    "TSMOM Portfolio": [
        f"{pm['Sharpe']:.2f}",
        f"{pm['Ann. Vol']:.1%}",
        f"{pm['Max DD']:.1%}",
        f"{pm['Calmar']:.2f}",
        f"{pm['Hit Rate']:.1%}",
        f"{pm['Ann. Turnover']:.1f}x",
    ],
}).set_index("Metric")

print("=" * 65)
print("  CANONICAL TSMOM vs MEMORY FILE TARGETS  (IS: 2015-2022)")
print("=" * 65)
comparison

## Export

Save signals, weights, equity, bucket risk, and summary.

In [ ]:
output_dir = Path("../outputs")
output_dir.mkdir(exist_ok=True)

datestamp = datetime.now().strftime("%Y%m%d")

# 1. Monthly signals
sig_path = output_dir / f"tsmom_signals_monthly_{datestamp}.csv"
signals.rename(columns=LABELS).to_csv(sig_path)

# 2. Daily weights (pre-overlay)
w_path = output_dir / f"tsmom_weights_daily_{datestamp}.csv"
weights_scaled.rename(columns=LABELS).to_csv(w_path)

# 3. Final daily weights (post-overlay)
wf_path = output_dir / f"tsmom_weights_final_daily_{datestamp}.csv"
final_weights.rename(columns=LABELS).to_csv(wf_path)

# 4. Portfolio equity
eq_path = output_dir / f"tsmom_portfolio_equity_{datestamp}.csv"
eq_export = pd.DataFrame({
    "portfolio_equity": port_equity,
    "portfolio_ret_net": port_ret_net,
    "portfolio_ret_gross": port_ret_gross,
    "turnover": turnover_daily,
})
eq_export.to_csv(eq_path)

# 5. Bucket risk shares
br_path = output_dir / f"tsmom_bucket_risk_shares_{datestamp}.csv"
bucket_risk_shares.to_csv(br_path)

# 6. Summary HTML
html_path = output_dir / f"tsmom_summary_{datestamp}.html"
html_content = (
    "<h2>Canonical TSMOM — IS Performance (2015-2022)</h2>\n"
    "<p>Signal: sign(12M return) | Monthly rebalance | EWMA vol (lambda=0.94) | "
    f"Commodity budget: {COMMODITY_RISK_BUDGET:.0%} | Portfolio vol target: {PORTFOLIO_VOL_TARGET:.0%}</p>\n"
    "<h3>Per-instrument + Portfolio Metrics</h3>\n"
    + fmt_df.to_html()
    + "<br><h3>vs Memory File Targets (section 6.3)</h3>\n"
    + comparison.to_html()
)
with open(html_path, "w") as f:
    f.write(html_content)

print(f"Exported to {output_dir.resolve()}/")
print(f"  {sig_path.name:45s}  ({signals.shape[0]} rows x {signals.shape[1]} cols)")
print(f"  {w_path.name:45s}  ({weights_scaled.shape[0]} rows)")
print(f"  {wf_path.name:45s}  ({final_weights.shape[0]} rows)")
print(f"  {eq_path.name:45s}  ({len(port_equity)} rows)")
print(f"  {br_path.name:45s}  ({len(bucket_risk_shares)} rows)")
print(f"  {html_path.name}")
print(f"\nNotebook complete: {datetime.now():%Y-%m-%d %H:%M}")